In [ ]:
import matplotlib.pyplot as plt
import torch 
import torch.distributions as dist
import math
torch.manual_seed(18081997)

# ============================================================
# 1. Función verdadera
# ============================================================

def f(x):
    return 1 + 2*x + 3*x**2


# ============================================================
# 2. Configuración de la simulación
# ============================================================

B = 1000       # número de muestras de entrenamiento
n = 10         # observaciones por muestra
sigma = 1.0    # desviación estándar del ruido

# Puntos donde queremos estudiar las predicciones
x_test = torch.linspace(-1, 1, 100)

# Valor verdadero de f(x)
y_true = f(x_test)


# ============================================================
# 3. Guardamos las predicciones de cada modelo
# ============================================================

pred_linear = torch.zeros(B, len(x_test))
pred_poly10 = torch.zeros(B, len(x_test))


# ============================================================
# 4. Repetimos el experimento B veces
# ============================================================

for b in range(B):

    # --------------------------------------------
    # Generar una nueva muestra de entrenamiento
    # --------------------------------------------

    x_train = 2 * torch.rand(n) - 1

    epsilon = sigma * torch.randn(n)

    y_train = f(x_train) + epsilon

    # --------------------------------------------
    # Modelo lineal
    # --------------------------------------------

    X_linear = torch.column_stack([
        torch.ones(n),
        x_train
    ])

    beta_linear = torch.linalg.lstsq(
        X_linear,
        y_train
    ).solution

    X_test_linear = torch.column_stack([
        torch.ones(len(x_test)),
        x_test
    ])

    pred_linear[b] = X_test_linear @ beta_linear

    # --------------------------------------------
    # Polinomio grado 10
    # --------------------------------------------

    X_poly10 = torch.stack(
        [x_train**j for j in range(11)],
        dim=1
    )

    beta_poly10 = torch.linalg.lstsq(
        X_poly10,
        y_train
    ).solution

    X_test_poly10 = torch.stack(
        [x_test**j for j in range(11)],
        dim=1
    )

    pred_poly10[b] = X_test_poly10 @ beta_poly10


# ============================================================
# 5. Predicción promedio
# ============================================================

mean_linear = pred_linear.mean(dim=0)
mean_poly10 = pred_poly10.mean(dim=0)


# ============================================================
# 6. Bias^2
# ============================================================

bias2_linear = (mean_linear - y_true)**2
bias2_poly10 = (mean_poly10 - y_true)**2


# ============================================================
# 7. Varianza
# ============================================================

variance_linear = ((pred_linear - mean_linear)**2).mean(dim=0)
variance_poly10 = ((pred_poly10 - mean_poly10)**2).mean(dim=0)


# ============================================================
# 8. Error cuadrático esperado
# ============================================================

mse_linear = ((pred_linear - y_true)**2).mean(dim=0)
mse_poly10 = ((pred_poly10 - y_true)**2).mean(dim=0)


# ============================================================
# 9. Gráfica de las predicciones
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    x_test,
    y_true,
    linewidth=3,
    label="Función verdadera"
)

# Algunas de las curvas obtenidas
for b in range(20):
    plt.plot(
        x_test,
        pred_linear[b],
        alpha=0.2
    )

plt.plot(
    x_test,
    mean_linear,
    linewidth=3,
    label="Predicción promedio (lineal)"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Modelo lineal")
plt.legend()
plt.show()


# ============================================================
# 10. Polinomio grado 10
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    x_test,
    y_true,
    linewidth=3,
    label="Función verdadera"
)

for b in range(20):
    plt.plot(
        x_test,
        pred_poly10[b],
        alpha=0.2
    )

plt.plot(
    x_test,
    mean_poly10,
    linewidth=3,
    label="Predicción promedio (grado 10)"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Polinomio grado 10")
plt.legend()
plt.ylim(-5, 10)
plt.show()


# ============================================================
# 11. Bias² y Varianza
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    x_test,
    bias2_linear,
    linewidth=2,
    label="Bias² - lineal"
)

plt.plot(
    x_test,
    variance_linear,
    linewidth=2,
    label="Varianza - lineal"
)

plt.plot(
    x_test,
    bias2_poly10,
    linewidth=2,
    label="Bias² - grado 10"
)

plt.plot(
    x_test,
    variance_poly10,
    linewidth=2,
    label="Varianza - grado 10"
)

plt.xlabel("x")
plt.ylabel("Error")
plt.title("Bias-Variance")
plt.legend()
plt.show()


# ============================================================
# 12. Promedios sobre todos los x
# ============================================================

print("MODELO LINEAL")
print("-----------------------------")
print("Bias² promedio:   ", bias2_linear.mean().item())
print("Variance promedio:", variance_linear.mean().item())
print("MSE promedio:     ", mse_linear.mean().item())


print("\nPOLINOMIO GRADO 10")
print("-----------------------------")
print("Bias² promedio:   ", bias2_poly10.mean().item())
print("Variance promedio:", variance_poly10.mean().item())
print("MSE promedio:     ", mse_poly10.mean().item())

In [ ]:
torch.manual_seed(18081997)


# ============================================================
# 1. Función verdadera
# ============================================================

def f(x):
    return 1 + 2*x + 3*x**2


# ============================================================
# 2. Parámetros
# ============================================================

B = 1000       # número de muestras de entrenamiento
n = 10         # observaciones por muestra
sigma = 1.0    # desviación estándar del ruido

x0 = torch.tensor(0.5)

# Verdadero valor de f(x0)
true_value = f(x0)

print("f(x0) =", true_value.item())


# ============================================================
# 3. Guardamos las predicciones
# ============================================================

pred_linear = torch.zeros(B)
pred_poly10 = torch.zeros(B)


# ============================================================
# 4. Simulación
# ============================================================

for b in range(B):

    # --------------------------------------------------------
    # Generamos una nueva muestra de entrenamiento
    # --------------------------------------------------------

    x_train = 2 * torch.rand(n) - 1

    epsilon = sigma * torch.randn(n)

    y_train = f(x_train) + epsilon


    # ========================================================
    # MODELO LINEAL
    # ========================================================

    X_linear = torch.column_stack([
        torch.ones(n),
        x_train
    ])

    beta_linear = torch.linalg.lstsq(
        X_linear,
        y_train
    ).solution

    x0_linear = torch.tensor([
        [1.0, x0.item()]
    ])

    pred_linear[b] = (
        x0_linear @ beta_linear
    ).squeeze()


    # ========================================================
    # POLINOMIO GRADO 10
    # ========================================================

    X_poly = torch.stack(
        [x_train**j for j in range(11)],
        dim=1
    )

    beta_poly = torch.linalg.lstsq(
        X_poly,
        y_train
    ).solution

    x0_poly = torch.tensor([
        [x0**j for j in range(11)]
    ])

    pred_poly10[b] = (
        x0_poly @ beta_poly
    ).squeeze()


# ============================================================
# 5. Bias y variance
# ============================================================

mean_linear = pred_linear.mean()
mean_poly10 = pred_poly10.mean()

bias2_linear = (mean_linear - true_value)**2
bias2_poly10 = (mean_poly10 - true_value)**2

variance_linear = (
    (pred_linear - mean_linear)**2
).mean()

variance_poly10 = (
    (pred_poly10 - mean_poly10)**2
).mean()


# ============================================================
# 6. Imprimir resultados
# ============================================================

print("\n==============================")
print("MODELO LINEAL")
print("==============================")

print("Predicción promedio:",
      mean_linear.item())

print("Bias²:",
      bias2_linear.item())

print("Varianza:",
      variance_linear.item())


print("\n==============================")
print("POLINOMIO GRADO 10")
print("==============================")

print("Predicción promedio:",
      mean_poly10.item())

print("Bias²:",
      bias2_poly10.item())

print("Varianza:",
      variance_poly10.item())


# ============================================================
# 7. Histogramas
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


# ------------------------------------------------------------
# Modelo lineal
# ------------------------------------------------------------

axes[0].hist(
    pred_linear.numpy(),
    bins=30
)

axes[0].axvline(
    true_value.item(),
    linewidth=3,
    label=r"$f(x_0)$ verdadero"
    ,color='r'
)

axes[0].axvline(
    mean_linear.item(),
    linestyle="--",
    linewidth=2,
    label=r"$E[\hat f(x_0)]$"
    ,color='r'
)

axes[0].set_title(
    "Modelo lineal"
)

axes[0].set_xlabel(
    r"$\hat f(x_0)$"
)

axes[0].set_ylabel(
    "Frecuencia"
)

axes[0].legend()


# ------------------------------------------------------------
# Polinomio grado 10
# ------------------------------------------------------------

axes[1].hist(
    pred_poly10.numpy(),
    bins=30
)

axes[1].axvline(
    true_value.item(),
    linewidth=3,
    label=r"$f(x_0)$ verdadero",
    color='r'
)

axes[1].axvline(
    mean_poly10.item(),
    linestyle="--",
    linewidth=2,
    label=r"$E[\hat f(x_0)]$"
    ,color='r'
)

axes[1].set_title(
    "Polinomio grado 10"
)

axes[1].set_xlabel(
    r"$\hat f(x_0)$"
)

axes[1].set_ylabel(
    "Frecuencia"
)

axes[1].legend()


plt.tight_layout()
plt.show()